# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** Two audits in one notebook: (1) two
findings from FlyRank's March 2026 research paper, read with a methodology skeptic's eye, and
(2) my own Week-5 model (`w05_model.ipynb`), re-run under a naive split to show exactly how
much a bad split would have inflated it.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from,
and does the validation design carry the claim? Constructive tone.*

### Finding #4 — "The Freshness Multiplier" (`docs/flyrank-seo-research-march-2026.pdf`, p.8)

**The claim:** "365+ day content that was refreshed within 30 days shows 3.2x health boost
(from 10.7 to 34.5) and 57x more impressions... refresh timing is one of the strongest measured
levers available."

**Where the comparison comes from:** two groups of pages — old content that was recently
refreshed, and old content that wasn't — compared on their current health/impression stats.

**My methodology question:** is this a *before/after* comparison on the same pages
(longitudinal — this page's health went from 10.7 to 34.5 after its own refresh), or a
*refreshed vs. not-refreshed* comparison across two different sets of pages (cross-sectional)?
The paper's phrasing ("from 10.7 to 34.5") reads like a within-page before/after, but nothing
in the write-up states whether the same `content_id`s were tracked pre- and post-refresh, or
whether "refreshed" pages were simply a different, possibly pre-selected population — teams
tend to choose *which* mature pages to refresh based on proven demand, which would make
already-stronger pages more likely to get refreshed in the first place. That's a selection
effect, not a refresh effect, and the write-up can't rule it out either way. The paper itself
is honest about a *related* stats problem in the same finding — the 361+ day growth-ratio
bucket hits 283:1 only because it has a single declining page, and the paper flags that
explicitly. It would strengthen the refresh-boost number to apply the same skepticism: state
the sample size behind the 10.7→34.5 comparison and say plainly whether it's the same pages
tracked over time or two different cohorts.

### ML Appendix — "What Predicts Growth?" (p.28)

**The claim:** a logistic regression gets "71% holdout accuracy," and "Content Age is the
strongest negative signal" toward decline, ahead of days-since-update and days-visible.

**Where the label comes from:** growing vs. declining pages — the same `trend_direction`-style
split my own `is_declining_label` uses.

**My methodology question:** the validation design behind that 71% isn't described — no split
strategy (random vs. grouped by client), no base rate next to the accuracy number, and the
"strongest signal" ranking is a bar chart with no coefficient values or units, just relative
bar size. Comparing this to my own Week-5 model directly (section 2-3 below): my grouped,
client-holdout split lands at AUC 0.616 with a printed base rate; a naive row-level split on
the same data pushes AUC to 0.711 purely from client memorization. If the paper's 71% accuracy
came from a random (not client-grouped) holdout, some of that number could be the same
inflation I can now show directly rather than genuine predictive signal — I can't tell from
the write-up, and that's exactly the gap a stated split design would close. Worth noting
constructively: my own top permutation-importance feature is `log_impressions_90d`, with
`content_age_days` third — a different top signal than the paper's chart, which doesn't
necessarily mean either is wrong (different feature sets, different label thresholds), but
it's a good reason not to treat either ranking as the final word without checking the split.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

ML-08 already used a **grouped-by-`client_id`** split from the start — the "after." To make
the gap visible rather than assumed, this section builds the "before": the exact same
Logistic Regression pipeline, same features, same random seed, but split at the row level
instead of by client.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct',
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna('unknown').astype(str)
for c in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    df[f'log_{c}'] = np.log1p(df[c])
NUM_FINAL = [c for c in NUMERIC_FEATURES if c not in
             ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']] + \
            ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']

X = df[NUM_FINAL + CATEGORICAL_FEATURES]
y = df['is_declining_label']
groups = df['client_id']

pre = ColumnTransformer([
    ('num', StandardScaler(), NUM_FINAL),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order[:k]].mean())

Ks = [20, 50, 100, 200]

In [2]:
# --- BEFORE: naive random row-level split ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe_random = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe_random.fit(X_tr, y_tr)
scores_random = pipe_random.predict_proba(X_te)[:, 1]
auc_random = roc_auc_score(y_te, scores_random)

train_clients_random = set(df.loc[X_tr.index, 'client_id'])
test_clients_random  = set(df.loc[X_te.index, 'client_id'])
overlap_random = train_clients_random & test_clients_random

print('BEFORE -- naive random row-level split:')
print(f'  AUC: {auc_random:.4f}')
for k in Ks:
    print(f'  precision@{k}: {precision_at_k(y_te.values, scores_random, k):.3f}')
print(f'  base rate (test): {y_te.mean():.3f}')
print(f'  clients appearing in BOTH train and test: {len(overlap_random)} of {df["client_id"].nunique()} (this is the leak)')

BEFORE -- naive random row-level split:
  AUC: 0.7106
  precision@20: 0.900
  precision@50: 0.920
  precision@100: 0.890
  precision@200: 0.870
  base rate (test): 0.542
  clients appearing in BOTH train and test: 31 of 32 (this is the leak)


In [3]:
# --- AFTER: grouped-by-client_id split (what ML-08 actually used) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

pipe_grouped = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe_grouped.fit(X.iloc[train_idx], y.iloc[train_idx])
scores_grouped = pipe_grouped.predict_proba(X.iloc[test_idx])[:, 1]
y_test_grouped = y.iloc[test_idx].values
auc_grouped = roc_auc_score(y_test_grouped, scores_grouped)

train_clients_grouped = set(groups.iloc[train_idx])
test_clients_grouped  = set(groups.iloc[test_idx])

print('AFTER -- grouped by client_id split:')
print(f'  AUC: {auc_grouped:.4f}')
for k in Ks:
    print(f'  precision@{k}: {precision_at_k(y_test_grouped, scores_grouped, k):.3f}')
print(f'  base rate (test): {y_test_grouped.mean():.3f}')
print(f'  clients appearing in both: {len(train_clients_grouped & test_clients_grouped)} (must be 0)')
print()
print(f'Gap (random - grouped) AUC: {auc_random - auc_grouped:+.4f}')

AFTER -- grouped by client_id split:
  AUC: 0.6157
  precision@20: 0.700
  precision@50: 0.720
  precision@100: 0.700
  precision@200: 0.710
  base rate (test): 0.511
  clients appearing in both: 0 (must be 0)

Gap (random - grouped) AUC: +0.0949


**Reading the gap:** the naive row-level split overstates AUC by **+0.095** (0.711 vs. 0.616)
and precision@20 by 20 points (0.900 vs. 0.700) — because 31 of the 32 clients leak into both
train and test, letting the model partly memorize per-client baselines instead of learning a
transferable decline signal. This is precisely the ML Appendix question from section 1: an
unstated split design can hide exactly this much inflation, and there's no way to tell from a
single reported accuracy number whether it happened.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Product-flag scan on the exact features the model trains on (NUM_FINAL + CATEGORICAL_FEATURES)
FLAG_PATTERNS = ('flag', 'score', 'priority', 'needs_', 'health', 'action_type')
suspects = [c for c in NUM_FINAL + CATEGORICAL_FEATURES if any(p in c.lower() for p in FLAG_PATTERNS)]
print(f'Product-flag-pattern scan on final feature list: {suspects}')
assert not suspects, 'a product-flag-like column made it into the feature list'
print('None found.')

Product-flag-pattern scan on final feature list: []
None found.


In [5]:
# Label-derived-feature trap, on the FINAL feature set (not just ML-04's toy 5 features):
# add trend_pct and impressions_last_30d -- the raw inputs trend_direction/is_declining_label
# are computed FROM -- and watch the honest 0.616 AUC collapse toward perfect.
X2 = df[NUM_FINAL + CATEGORICAL_FEATURES + ['trend_pct', 'impressions_last_30d']].copy()
X2['trend_pct']             = pd.to_numeric(X2['trend_pct'], errors='coerce').fillna(0)
X2['impressions_last_30d']  = pd.to_numeric(X2['impressions_last_30d'], errors='coerce').fillna(0)

pre2 = ColumnTransformer([
    ('num', StandardScaler(), NUM_FINAL + ['trend_pct', 'impressions_last_30d']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])
pipe_leak = Pipeline([('pre', pre2), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe_leak.fit(X2.iloc[train_idx], y.iloc[train_idx])
auc_leak = roc_auc_score(y.iloc[test_idx], pipe_leak.predict_proba(X2.iloc[test_idx])[:, 1])

print(f'Honest AUC (final feature set, grouped split): {auc_grouped:.4f}')
print(f'Leaked AUC (+ trend_pct, impressions_last_30d): {auc_leak:.4f}')
print(f'Jump: +{auc_leak - auc_grouped:.4f}  <- confession. These two columns never enter the real model.')

Honest AUC (final feature set, grouped split): 0.6157
Leaked AUC (+ trend_pct, impressions_last_30d): 0.9985
Jump: +0.3828  <- confession. These two columns never enter the real model.


In [6]:
# Population selection check + base rate, printed next to every metric per the skill.
print(f'Population: {len(df):,} of {len(df):,} rows used -- no row filter applied before')
print('modeling, so which rows exist in the feature set cannot depend on the outcome window.')
print()
print(f'Base rate (full dataset): {y.mean():.3f}')
print(f'Base rate (grouped test split): {y.iloc[test_idx].mean():.3f}')
print()
print('Attack checklist, final feature set:')
print('  [x] Timeline: 90-day trailing features vs. 30d-vs-prev-30d label -- feature window')
print('      predates the comparison window used to build trend_direction')
print('  [x] No label-derived columns in the feature set (confirmed above via the trap test)')
print('  [x] No product-flag columns in the feature set (confirmed above via the pattern scan)')
print('  [x] Population selection carries no outcome-window information (full 30,000 rows used)')
print('  [x] Split grouped by client_id (Section 2)')
print('  [x] Base rate printed next to every metric')
print('  [x] Top feature importance sanity-checked in ML-08 (traffic/age -- nothing suspicious)')

Population: 30,000 of 30,000 rows used -- no row filter applied before
modeling, so which rows exist in the feature set cannot depend on the outcome window.

Base rate (full dataset): 0.542
Base rate (grouped test split): 0.511

Attack checklist, final feature set:
  [x] Timeline: 90-day trailing features vs. 30d-vs-prev-30d label -- feature window
      predates the comparison window used to build trend_direction
  [x] No label-derived columns in the feature set (confirmed above via the trap test)
  [x] No product-flag columns in the feature set (confirmed above via the pattern scan)
  [x] Population selection carries no outcome-window information (full 30,000 rows used)
  [x] Split grouped by client_id (Section 2)
  [x] Base rate printed next to every metric
  [x] Top feature importance sanity-checked in ML-08 (traffic/age -- nothing suspicious)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured,
directional, decision-support.*

In [7]:
boldest_original = (
    "Logistic Regression wins on every metric here -- and Random Forest, despite being the "
    "'stronger' model in the toolkit's table, does not beat it."
)
safer_rewrite = (
    "On this dataset's client-grouped holdout split, Logistic Regression showed the strongest "
    "observed ROC-AUC and precision@K among the three methods tested; Random Forest did not "
    "measurably improve on it here. This describes performance on this specific split and "
    "feature set, not a general claim that logistic regression outperforms random forests "
    "for content-decline prediction."
)
print('Original (from ML-08, w05_model.ipynb, section 3):')
print(f'  "{boldest_original}"')
print()
print('Rewritten:')
print(f'  "{safer_rewrite}"')

Original (from ML-08, w05_model.ipynb, section 3):
  "Logistic Regression wins on every metric here -- and Random Forest, despite being the 'stronger' model in the toolkit's table, does not beat it."

Rewritten:
  "On this dataset's client-grouped holdout split, Logistic Regression showed the strongest observed ROC-AUC and precision@K among the three methods tested; Random Forest did not measurably improve on it here. This describes performance on this specific split and feature set, not a general claim that logistic regression outperforms random forests for content-decline prediction."


**Why the rewrite matters:** "wins on every metric" reads as a general, model-class-level
claim ("logistic regression beats random forests"), which this single experiment cannot
support — it's one dataset, one feature set, one split, one seed. The rewrite scopes the claim
down to exactly what was measured (this split, these features) and states plainly that it
isn't a general result — the same discipline the flag-linked test in ML-06 already used
("a directional signal ... not proof the flag is wrong"), applied here to my own modeling
claim rather than someone else's flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.